# Final report

This notebook combines the former `1_data_overview_and_preprocessing`, `2_data_analysis` and `3_similarity` notebooks, in that order:

1. **Data overview and preprocessing:** inspects `data/catalog.json` and writes `data/features_en.json` and `data/products_en.json`.
2. **Data analysis:** statistics on the preprocessed feature names and products.
3. **Similarity matrix:** builds the shared similarity matrix and saves it to `data/similarity.npz` for all clustering notebooks.

Run it from the repository root. Part 3 downloads the Qwen embedding model (~1.2 GB) on first run.

# 1. Data overview and preprocessing

This notebook explores the raw `data/catalog.json` received from the company. We inspect its structure, product mix, feature-name frequency, and a reproducible sample of products and their feature descriptions. The examples use the catalog's default locale.

## 1.1 Load the catalog and inspect its structure

The catalog is nested: product fields live under `data`, localized text lives under `data.content[locale]`, and product features are stored as two parallel lists: `feature_titles` (the feature names) and `features` (the descriptions). The path check supports running this notebook either from this project folder or its parent workspace.

In [1]:
import html
import json
import random
from collections import Counter
from pathlib import Path
from pprint import pprint

catalog_path = Path("data/catalog.json")
if not catalog_path.is_file():
    catalog_path = Path("CTFDS-Project/data/catalog.json")
if not catalog_path.is_file():
    raise FileNotFoundError("Could not find data/catalog.json from the notebook or workspace folder.")

catalog = json.loads(catalog_path.read_text(encoding="utf-8"))
products = catalog.get("products")
if not isinstance(products, list):
    raise ValueError("Expected catalog.json to contain a 'products' list.")

default_locale = catalog.get("default_locale", "en-GB")
print(f"Catalog: {catalog_path.resolve()}")
print(f"Schema version: {catalog.get('schema_version')}")
print(f"Products: {len(products)}")
print(f"Default locale: {default_locale}")
print(f"Locales: {catalog.get('locales', [])}")
print("Top-level keys:", list(catalog.keys()))
print("Keys in a product record:", list(products[0].keys()) if products else [])

Catalog: /sessions/rcw-01u1bd41xueqhlvmvih7pz9u/work/tmp/data/catalog.json
Schema version: 1.0.0
Products: 208
Default locale: en-GB
Locales: ['en-GB', 'de-DE', 'es-ES', 'fr-FR', 'it-IT', 'nl-NL', 'sv-SE', 'da-DK', 'nb-NO']
Top-level keys: ['schema_version', 'default_locale', 'locales', 'currencies', 'inheritance', 'products']
Keys in a product record: ['id', 'data', 'media', 'variants']


## 1.2 Product and feature statistics

Feature-name counts below describe each parent product's default-locale content and are exact after trimming whitespace and decoding HTML entities; capitalization and wording variants remain separate. We also summarize variants and check whether each product has the same number of feature names and descriptions, since the raw file appears to associate these by list position.

In [2]:
product_types = Counter()
genders = Counter()
locale_coverage = Counter()
feature_name_counts = Counter()
variant_total = 0
products_with_variants = 0
variants_with_default_locale_content_override = 0
product_summaries = []
length_mismatches = []
blank_feature_descriptions = []

for product in products:
    variants = product.get("variants") or []
    variant_total += len(variants)
    products_with_variants += bool(variants)
    for variant in variants:
        override_content = ((variant.get("overrides") or {}).get("content") or {})
        variants_with_default_locale_content_override += default_locale in override_content

    product_data = product.get("data") or {}
    classification = product_data.get("classification") or {}
    content_by_locale = product_data.get("content") or {}
    locale_coverage.update(content_by_locale.keys())
    content = content_by_locale.get(default_locale) or {}
    feature_names = content.get("feature_titles") or []
    descriptions = content.get("features") or []

    product_type = classification.get("product_type", "[missing]")
    gender = classification.get("gender", "[missing]")
    product_types[product_type] += 1
    genders[gender] += 1

    clean_feature_names = [html.unescape(str(feature_name)).strip() for feature_name in feature_names]
    clean_feature_names = [feature_name for feature_name in clean_feature_names if feature_name]
    feature_name_counts.update(clean_feature_names)
    product_summaries.append({
        "id": product.get("id", "[missing]"),
        "name": content.get("name", "[missing name]"),
        "product_type": product_type,
        "gender": gender,
        "feature_count": len(feature_names),
        "description_count": len(descriptions),
    })

    if len(feature_names) != len(descriptions):
        length_mismatches.append({
            "id": product.get("id", "[missing]"),
            "name": content.get("name", "[missing name]"),
            "feature_names": len(feature_names),
            "descriptions": len(descriptions),
        })
    for index, feature_name in enumerate(feature_names):
        if index >= len(descriptions) or not str(descriptions[index]).strip():
            blank_feature_descriptions.append((product.get("id"), feature_name))

print(f"Parent products: {len(products)}")
print(f"Variants: {variant_total} across {products_with_variants} products")
print(f"Variants with a {default_locale} content override: {variants_with_default_locale_content_override}")
print(f"Localized product-content records: {dict(sorted(locale_coverage.items()))}")
print(f"Feature-name occurrences: {sum(feature_name_counts.values())}")
print(f"Distinct exact feature names: {len(feature_name_counts)}")
print(f"Products with different feature-name/description counts: {len(length_mismatches)}")
print(f"Feature names without a non-empty description at the same position: {len(blank_feature_descriptions)}")

print("\nProducts by type:")
for name, count in product_types.most_common():
    print(f"  {name}: {count}")

print("\nProducts by gender:")
for name, count in genders.most_common():
    print(f"  {name}: {count}")

print("\nMost frequent feature names:")
for feature_name, count in feature_name_counts.most_common(15):
    print(f"  {count:>3}  {feature_name}")

Parent products: 208
Variants: 444 across 208 products
Variants with a en-GB content override: 72
Localized product-content records: {'da-DK': 208, 'de-DE': 208, 'en-GB': 208, 'es-ES': 208, 'fr-FR': 208, 'it-IT': 208, 'nb-NO': 208, 'nl-NL': 208, 'sv-SE': 208}
Feature-name occurrences: 1073
Distinct exact feature names: 444
Products with different feature-name/description counts: 7
Feature names without a non-empty description at the same position: 13

Products by type:
  Cycling Wear Tops: 42
  Cycling Wear Bottoms: 31
  Socks: 25
  Headwear: 24
  Shoe Covers: 21
  Base Layers: 21
  Gloves Cycling Winter: 15
  Warmers: 12
  Gloves Cycling Summer Short: 7
  Gloves Cycling Summer Long: 6
  Accessories: 2
  Cycling Wear One-Pieces: 2

Products by gender:
  Unisex: 115
  Men's: 54
  Women's: 38
  Junior: 1

Most frequent feature names:
   53  Reflective details
   23  Highly breathable
   15  Durable water repellent (DWR) treatment
   15  Stretch arch support
   14  Anti-odour treatment
  

## 1.3 Inspect sample products and features

The sample is reproducible and includes a product with mismatched feature-list lengths when one exists. If the two lists have different lengths, the code prints them separately rather than using `zip`, which would silently drop unmatched entries and imply pairings we cannot verify.

In [3]:
sample_products = random.Random(42).sample(products, k=min(3, len(products)))
mismatch_ids = {row["id"] for row in length_mismatches}
mismatch_example = next((product for product in products if product.get("id") in mismatch_ids), None)
if mismatch_example is not None and all(p.get("id") != mismatch_example.get("id") for p in sample_products):
    sample_products.append(mismatch_example)

for product in sample_products:
    product_data = product.get("data") or {}
    classification = product_data.get("classification") or {}
    content = (product_data.get("content") or {}).get(default_locale) or {}
    feature_names = content.get("feature_titles") or []
    descriptions = content.get("features") or []

    print(f"\nProduct {product.get('id')} — {content.get('name', '[missing name]')}")
    print(f"Type: {classification.get('product_type', '[missing]')} | Gender: {classification.get('gender', '[missing]')}")
    variants = product.get("variants") or []
    print(f"Variants: {len(variants)}")
    for variant in variants[:2]:
        color = (variant.get("color") or {}).get("name", "[color not specified]")
        print(f"  Variant {variant.get('id', '[missing id]')} | color: {color}")
    if len(feature_names) == len(descriptions):
        for feature_name, description in zip(feature_names, descriptions):
            print(f"  - {feature_name}: {description or '[empty description]'}")
    else:
        print(f"  Feature names ({len(feature_names)}):")
        for feature_name in feature_names:
            print(f"    - {feature_name}")
        print(f"  Descriptions ({len(descriptions)}; pairing not assumed):")
        for description in descriptions:
            print(f"    - {description or '[empty description]'}")


Product 6111 — GripGrab RIDE Cycling Shorts
Type: Cycling Wear Bottoms | Gender: Men's
Variants: 1
  Variant 611101 | color: Black
  - High-quality construction: Crafted from high-quality fabrics offering a balanced level of stretch and support for enhanced comfort
  - Quality padding (LIEGE HP): Integrates a high-quality cycling pad from Elastic Interface (LIEGE HP) to deliver outstanding comfort during long rides
  - 1 top entry leg pocket: Features a convenient stretchy pocket, lying flat against the leg for easy access to your phone or snacks throughout your ride
  - Wide elastic hems: Ensure a secure fit on the legs with wide elastic hems and silicone grippers
  - Wicking waistband: Stays comfortably in place during your rides thanks to the stretch waistband lined with moisture-wicking mesh

Product 1075 — GripGrab Primavera 2 Merino Spring-Autumn Gloves
Type: Gloves Cycling Winter | Gender: Unisex
Variants: 1
  Variant 107501 | color: Black
  - Luxurious soft merino blend: A mer

In [4]:
print("Examples for the three most frequent feature names:")
for feature_name, _ in feature_name_counts.most_common(3):
    print(f"\n{feature_name}")
    shown = 0
    for product in products:
        product_data = product.get("data") or {}
        content = (product_data.get("content") or {}).get(default_locale) or {}
        feature_names = content.get("feature_titles") or []
        descriptions = content.get("features") or []
        if len(feature_names) != len(descriptions):
            continue
        for raw_feature_name, description in zip(feature_names, descriptions):
            if html.unescape(str(raw_feature_name)).strip() == feature_name:
                print(f"  {content.get('name', '[missing name]')}: {description or '[empty description]'}")
                shown += 1
                if shown == 3:
                    break
        if shown == 3:
            break

Examples for the three most frequent feature names:

Reflective details
  GripGrab Waterproof Helmet Cover: for visibility in the traffic
  GripGrab W's PACR Thermal Bib Shorts: For visibility in traffic
  GripGrab PACR Windproof Lightweight Vest: Enhance your visibility to other road users in low light conditions

Highly breathable
  GripGrab FastStream Aero Cycling Socks: The foot section of the sock, crafted from Coolmax® yarn, offers great breathability
  GripGrab RIDE Winter Long Sleeve Base Layer: providing excellent ventilation
  GripGrab Ultralight Mesh Sleeveless Base Layer 3-Pack: Ultralight open mesh fabric allows heat and moisture to escape from within, keeping you cool in hot summer conditions

Durable water repellent (DWR) treatment
  GripGrab W's PACR Thermal Bib Shorts: Providing effective water repellency
  GripGrab RIDE 2 Waterproof Winter Gloves: providing effective water repellency
  GripGrab RIDE 2 Windproof Spring-Autumn Gloves: providing effective water repellenc

## 1.4 Why preprocessing is needed and what it will do

The raw catalog is useful for product-level inspection, but it is not yet a clean feature table. Content is nested by product and locale; each product may also have variant-level overrides. Feature names and descriptions are stored in separate positional lists, and some products have different list lengths or empty descriptions. Repeated feature names also occur across products with different wording, capitalization, or HTML entities.

We will turn the catalog into two smaller files for the analysis that follows:

- `data/features_en.json`: one record per cleaned feature name, with its associated descriptions; descriptions that cannot safely be paired are kept separately.
- `data/products_en.json`: one record per product, with its product type and feature names, for later evaluation.

The JSON files keep the catalog's field names (`attribute_records`, `title`, `titles`); in the text we call these feature names.

The code selects the default English locale, includes product-level and variant-override content, decodes HTML entities, trims text, and aggregates repeated feature names. When a product's feature-name and description lists have different lengths, it does not guess the pairings: it keeps those descriptions as unpaired instead of silently losing them with `zip`.

## 1.5 Preprocess the catalog

Run this after the overview cells. The outputs are written next to `catalog.json` and are used by the following analysis and clustering notebooks.

In [5]:
import html
import json

data_dir = catalog_path.parent
features_path = data_dir / "features_en.json"
products_path = data_dir / "products_en.json"

blocks = []
for raw_product in products:
    product_type = raw_product["data"]["classification"]["product_type"]
    product_id = raw_product["id"]
    content = (raw_product.get("data") or {}).get("content") or {}
    blocks.append((product_id, product_type, content.get(default_locale, {})))

    for variant in raw_product.get("variants") or []:
        variant_content = ((variant.get("overrides") or {}).get("content") or {})
        blocks.append((product_id, product_type, variant_content.get(default_locale, {})))

feature_name_records = {}
product_records = {}
unpaired_descriptions = []

for product_id, product_type, content in blocks:
    feature_names = [html.unescape(feature_name).strip() for feature_name in content.get("feature_titles") or []]
    descriptions = [html.unescape(description).strip() for description in content.get("features") or []]

    product = product_records.setdefault(
        product_id,
        {"product_id": product_id, "product_type": product_type, "titles": set()},
    )
    product["titles"].update(feature_names)

    for feature_name in feature_names:
        feature_name_records.setdefault(feature_name, {"title": feature_name, "descriptions": []})

    if len(feature_names) != len(descriptions):
        unpaired_descriptions.extend(descriptions)
        continue

    for feature_name, description in zip(feature_names, descriptions):
        feature_name_records[feature_name]["descriptions"].append(description)

for product in product_records.values():
    product["titles"] = sorted(product["titles"])

features_path.write_text(
    json.dumps(
        {
            "attribute_records": list(feature_name_records.values()),
            "unpaired_descriptions": unpaired_descriptions,
        },
        ensure_ascii=False,
        indent=1,
    ),
    encoding="utf-8",
)
products_path.write_text(
    json.dumps(list(product_records.values()), ensure_ascii=False, indent=1),
    encoding="utf-8",
)

print(f"Wrote {len(feature_name_records)} unique feature names to {features_path}")
print(f"Kept {len(unpaired_descriptions)} descriptions that could not be paired safely")
print(f"Wrote {len(product_records)} products to {products_path}")

Wrote 445 unique feature names to data/features_en.json
Kept 22 descriptions that could not be paired safely
Wrote 208 products to data/products_en.json


# 2. Data analysis

Do we get any insight after preprocessing?

We examine feature names per product, feature-name frequency, descriptions, and how often pairs of feature names co-occur on the same product.

In [6]:
import json
import statistics
from collections import Counter
from itertools import combinations
from pathlib import Path

records = json.loads(Path("data/features_en.json").read_text(encoding="utf8"))["attribute_records"]
products = json.loads(Path("data/products_en.json").read_text(encoding="utf8"))
print(len(records), "feature names,", len(products), "products")

# Feature names per product
per_product = [len(p["titles"]) for p in products]
print(f"\nFeature names per product: mean {statistics.mean(per_product):.1f}, median {statistics.median(per_product)}, "
      f"min {min(per_product)}, max {max(per_product)}")

# Feature-name frequency: on how many products each feature name appears
feature_name_counts = Counter(t for p in products for t in p["titles"])
print(f"\n{sum(1 for c in feature_name_counts.values() if c == 1)} of {len(feature_name_counts)} feature names appear on only one product")
print(f"{sum(1 for c in feature_name_counts.values() if c >= 5)} feature names appear on 5 or more products")

# Most common feature names
print("\nMost common feature names:")
for feature_name, count in feature_name_counts.most_common(10):
    print(f"{count:3}  {feature_name}")

# Descriptions
desc_counts = [len(r["descriptions"]) for r in records]
distinct_counts = [len(set(r["descriptions"])) for r in records]
print(f"\n{sum(1 for c in desc_counts if c == 0)} feature names have no description")
print(f"mean {statistics.mean(desc_counts):.1f} descriptions per feature name, but only {statistics.mean(distinct_counts):.1f} distinct")

# Pairs of feature names that appear on the same product 
# TODO (TO BE IMPROVED with APRIORI LOGIC)
same_product_pairs = set()
for p in products:
    same_product_pairs.update(combinations(p["titles"], 2))
n = len(records)
all_pairs = n * (n - 1) // 2
print(f"\nSame-product feature-name pairs: {len(same_product_pairs)} of {all_pairs} possible ({len(same_product_pairs) / all_pairs:.1%})")

445 feature names, 208 products

Feature names per product: mean 5.2, median 5.0, min 3, max 11

202 of 445 feature names appear on only one product
48 feature names appear on 5 or more products

Most common feature names:
 53  Reflective details
 23  Highly breathable
 15  Durable water repellent (DWR) treatment
 15  Stretch arch support
 14  Anti-odour treatment
 12  Reflective branding
 11  Race fit
 10  Moisture wicking
  9  COOLMAX® fibre
  9  Fast-drying material

12 feature names have no description
mean 2.3 descriptions per feature name, but only 1.4 distinct

Same-product feature-name pairs: 1442 of 98790 possible (1.5%)


## 2.1 Visualizing product and feature patterns

These plots explore the preprocessed catalog only: how many feature names products have, how often each feature name is reused across products, and how many description entries are collected for each feature name.

A feature name can have several description entries because it appears on several products. Some entries may contain identical text. The final plot counts entries, not unique wording; the examples below the chart show what those counts mean.

In [7]:
import plotly.graph_objects as go
from IPython.display import display

# How many distinct feature names are used by 1, 2, 3, 4, or 5+ products?
support_categories = ["1", "2", "3", "4", "5+"]
support_group = [str(count) if count < 5 else "5+" for count in feature_name_counts.values()]
support_category_counts = [sum(group == category for group in support_group) for category in support_categories]

support_figure = go.Figure(
    go.Bar(
        x=support_categories,
        y=support_category_counts,
        text=[f"{count} ({count / len(feature_name_counts):.1%})" for count in support_category_counts],
        textposition="outside",
        marker_color="#3d7899",
        hovertemplate="Appears on %{x} product(s)<br>Feature names: %{y}<extra></extra>",
    )
)
support_figure.update_layout(
    title="How often is each feature name used across products?",
    xaxis_title="Number of products containing a feature name",
    yaxis_title="Number of distinct feature names",
    template="simple_white",
    showlegend=False,
)
display(support_figure)

# Distribution of how many feature names each product has.
product_feature_name_counts = Counter(len(product["titles"]) for product in products)
product_feature_name_values = sorted(product_feature_name_counts)
product_figure = go.Figure(
    go.Bar(
        x=product_feature_name_values,
        y=[product_feature_name_counts[count] for count in product_feature_name_values],
        text=[product_feature_name_counts[count] for count in product_feature_name_values],
        textposition="outside",
        marker_color="#62a386",
        hovertemplate="Feature names per product: %{x}<br>Products: %{y}<extra></extra>",
    )
)
product_figure.update_layout(
    title="How many feature names does each product have?",
    xaxis_title="Number of feature names",
    yaxis_title="Number of products",
    template="simple_white",
    showlegend=False,
)
display(product_figure)

# Count description entries attached to each feature name.
description_categories = ["0", "1", "2", "3", "4", "5+"]
description_bins = Counter(str(count) if count < 5 else "5+" for count in desc_counts)

description_figure = go.Figure(
    go.Bar(
        x=description_categories,
        y=[description_bins[category] for category in description_categories],
        text=[description_bins[category] for category in description_categories],
        textposition="outside",
        marker_color="#d18b45",
        hovertemplate="Description entries for a feature name: %{x}<br>Feature names: %{y}<extra></extra>",
    )
)
description_figure.update_layout(
    title="How many description entries are attached to each feature name?",
    xaxis_title="Description entries per feature name (5+ grouped)",
    yaxis_title="Number of feature names",
    template="simple_white",
    showlegend=False,
)
display(description_figure)

# Concrete examples: show how repeated entries are stored for individual feature names.
example_feature_names = ["Light insulation", "Reflective details"]
for feature_name in example_feature_names:
    record = next((record for record in records if record["title"] == feature_name), None)
    if record is None:
        continue
    descriptions = Counter(record["descriptions"])
    print(
        f"\n{feature_name}: {sum(descriptions.values())} description entries, "
        f"{len(descriptions)} distinct text versions"
    )
    for description, count in descriptions.most_common(3):
        print(f"  {count}x: {description}")


Light insulation: 5 description entries, 3 distinct text versions
  3x: Delivering warmth and exceptional breathability for comfort during transitional rides
  1x: Soft knitted fabric offers warmth retention and protection from cooler spring-autumn temperatures and windchill
  1x: The Thermolite® fabric and padded sole provide light insulation for cool spring and autumn days

Reflective details: 53 description entries, 14 distinct text versions
  12x: for visibility in traffic
  9x: Enhance your visibility to other road users in low light conditions
  8x: For visibility in traffic


## 2.2 Example: feature names and descriptions for one product

In the raw catalog, each product has a list of feature names and a parallel list of descriptions. When the two lists have equal lengths, an item at the same position in each list is a feature-name/description pair. The example below prints one complete product record. Across the full catalog, the same feature name can occur on multiple products with different descriptions; preprocessing groups those description entries together by feature name.

In [8]:
import html
import json
from pathlib import Path

catalog_path = Path("data/catalog.json")
if not catalog_path.is_file():
    catalog_path = Path("CTFDS-Project/data/catalog.json")
if not catalog_path.is_file():
    raise FileNotFoundError("Could not find data/catalog.json.")

catalog = json.loads(catalog_path.read_text(encoding="utf-8"))
locale = catalog.get("default_locale", "en-GB")
sample = None
for product in catalog.get("products", []):
    product_data = product.get("data") or {}
    content = (product_data.get("content") or {}).get(locale) or {}
    feature_names = content.get("feature_titles") or []
    descriptions = content.get("features") or []
    if feature_names and len(feature_names) == len(descriptions):
        sample = (product, product_data, content, feature_names, descriptions)
        break

if sample is None:
    raise ValueError("No product with aligned feature names and descriptions was found.")

product, product_data, content, feature_names, descriptions = sample
classification = product_data.get("classification") or {}
print(f"Product {product.get('id')}: {content.get('name', '[missing name]')}")
print(f"Product type: {classification.get('product_type', '[missing]')}\n")

for feature_name, description in zip(feature_names, descriptions):
    print(f"Feature name: {html.unescape(str(feature_name)).strip()}")
    print(f"Description: {html.unescape(str(description)).strip()}\n")

Product 4023: GripGrab PACR Spring-Autumn Leg Warmers
Product type: Warmers

Feature name: Light insulation
Description: Delivering warmth and exceptional breathability for comfort during transitional rides

Feature name: DWR coating
Description: Delivering effective water repellency in wet conditions

Feature name: Pre-shaped fit
Description: Delivering a precise, anatomical fit that follows the natural contours of the legs

Feature name: Secure hold with elasticated grippers
Description: Keeping the leg warmers securely in place

Feature name: Reflective branding
Description: Adding a touch of reflectivity



# 3. Similarity matrix

We built one similarity score for every pair of the 445 feature names, used by all clustering notebooks.

1. Load `data/features_en.json` from Part 1 above.
2. TF-IDF similarity of the descriptions.
3. Semantic similarity from Qwen embeddings of feature name + descriptions.
4. Blend with fuzzy feature-name matching (50% semantic, 25% TF-IDF, 25% fuzzy).
5. Penalise pairs with conflicting concepts and save the matrix to `data/similarity.npz`.

**Input:** `data/features_en.json` from Part 1 above, with one record per unique feature name and its descriptions. The unpaired descriptions are only used for the TF-IDF word weights.

**Dependencies:** see `requirements.txt` (Python 3.13). The first run downloads the Qwen model (~1.2 GB).

In [1]:
import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
from rapidfuzz import fuzz, process, utils
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

# weights of the three similarity signals
W_SEMANTIC, W_TFIDF, W_FUZZY = 0.50, 0.25, 0.25
W_NAME = 0.50
PENALTY = 0.30
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
EMBEDDING_PROMPT = "Instruct: Identify product features that describe the same property\nQuery: "

W1121 23:07:26.430000 1366 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 3.1 Load the feature records

Loads the 445 feature names and their descriptions from `data/features_en.json`. The 22 unpaired descriptions are kept separately and only used for the TF-IDF word weights.

In [2]:
feature_data = json.loads(Path("data/features_en.json").read_text(encoding="utf8"))
feature_name_records = feature_data["attribute_records"]
unpaired_descriptions = feature_data["unpaired_descriptions"]

feature_names = [record["title"] for record in feature_name_records]
descriptions = [record["descriptions"] for record in feature_name_records]

print(len(feature_names), "feature names,", sum(bool(d) for d in descriptions), "with descriptions,",
      len(unpaired_descriptions), "unpaired descriptions")

445 titles, 433 with descriptions, 22 unpaired descriptions


## 3.2 TF-IDF similarity of the descriptions

Each feature name's descriptions are joined into one document. TF-IDF gives rare words more weight than words used everywhere. The word weights are learned from all descriptions but only feature names get a row in the matrix. Feature names without descriptions get 0 here and rely on the other two signals.

In [3]:
#here, each feature name has its document
documents = [" ".join(texts) for texts in descriptions]

# word weights are learned from all descriptions
tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True)
tfidf.fit([d for d in documents if d] + unpaired_descriptions)

# rows are normalised, so the dot product is the cosine similarity
vectors = tfidf.transform(documents)
tfidf_similarity = (vectors @ vectors.T).toarray()
print("TF-IDF similarity matrix:", tfidf_similarity.shape)

TF-IDF similarity matrix: (445, 445)


## 3.3 Semantic feature-name-and-description embeddings

An embedding is a vector (a point in a high-dimensional meaning space) produced from a piece of text. Texts with related meanings tend to have vectors pointing in similar directions, even when they use different words. This step creates one semantic vector for each feature name; it does not cluster the feature names yet.

For every feature name, the model embeds the feature name and each distinct, aligned description. If a feature name has several descriptions, their vectors are averaged using the number of times each description occurs in the data. The feature-name vector and this description-context vector are then blended equally (`W_NAME = 0.50`) and normalized. Later, the cosine similarity between two resulting vectors gives their raw semantic similarity.

**Why cosine similarity?** The vectors are normalized to length 1, so their dot product equals cosine similarity: `cos(theta) = (u · v) / (||u|| ||v||) = u · v`. Cosine compares vector direction (semantic orientation) rather than length. “Raw” means this score is calculated before the median-based calibration later in this notebook.

**Other possible comparisons:** Euclidean distance (`||u - v||`) is a natural alternative. For unit-length vectors, it gives the same ranking as cosine because `||u - v||^2 = 2(1 - cosine_similarity)`; only the distance scale changes. Manhattan distance (`sum(abs(u - v))`) is another option, but it measures separation differently and may rank pairs differently, so it should be checked against reviewed similar/dissimilar pairs of feature names. A plain dot product is also possible, but without normalization it is sensitive to vector length; with these normalized vectors it is identical to cosine similarity.

**Example — “Light insulation”:** this feature name has five description entries but three distinct texts. “Delivering warmth and exceptional breathability for comfort during transitional rides” occurs three times; the other two descriptions occur once each. The description-context vector is therefore conceptually `(3 × vector(description 1) + vector(description 2) + vector(description 3)) / 5`. That context is blended 50/50 with the embedding of the feature name “Light insulation”. The repeated description pulls the context toward its meaning, but does not change the feature name’s own embedding.

If a feature name has no aligned description, its semantic vector is just its own embedding—not a zero vector. Descriptions that could not safely be matched to feature names are excluded from this step. They are used only when fitting TF-IDF in the next section. The public model may need to download the first time; no access token or login is required.

In [4]:
import torch

embedding_model = SentenceTransformer(MODEL_NAME, model_kwargs={"dtype": torch.float32})
name_vectors = embedding_model.encode(
    feature_names,
    prompt=EMBEDDING_PROMPT,
    normalize_embeddings=True,
    show_progress_bar=True,
)

all_descriptions = sorted({text for texts in descriptions for text in texts})
description_embedding_vectors = embedding_model.encode(
    all_descriptions,
    prompt=EMBEDDING_PROMPT,
    normalize_embeddings=True,
    show_progress_bar=True,
)
description_vector_by_text = {
    text: description_embedding_vectors[index]
    for index, text in enumerate(all_descriptions)
}

feature_name_vectors = []
for name_vector, texts in zip(name_vectors, descriptions):
    if texts:
        description_counts = Counter(texts)
        weighted_description_vector = np.average(
            [description_vector_by_text[text] for text in description_counts],
            axis=0,
            weights=list(description_counts.values()),
        )
        vector = W_NAME * name_vector + (1 - W_NAME) * weighted_description_vector
        vector = vector / np.linalg.norm(vector)
    else:
        vector = name_vector
    feature_name_vectors.append(vector)

feature_name_vectors = np.asarray(feature_name_vectors)
raw_semantic = feature_name_vectors @ feature_name_vectors.T
baseline_similarity = np.median(raw_semantic[np.triu_indices(len(feature_names), k=1)])
if baseline_similarity >= 1:
    raise ValueError("Semantic similarities cannot be calibrated: median baseline is 1.")

# Calibrate model-specific cosine baselines to 0; identical vectors remain at 1.
semantic = np.clip(
    (raw_semantic - baseline_similarity) / (1 - baseline_similarity),
    0,
    1,
)
print(f"Semantic matrix ready; uncalibrated median baseline: {baseline_similarity:.3f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Semantic matrix ready; uncalibrated median baseline: 0.566


## 3.4 Blend the three similarities

Fuzzy matching compares the wording of the feature names, ignoring word order ("Cycling Gloves" = "Gloves Cycling"). The final similarity is a weighted average: 50% semantic, 25% TF-IDF and 25% fuzzy. We blend the similarity matrices instead of combining the vectors, because embeddings and TF-IDF vectors live in different spaces.

In [5]:
assert np.isclose(W_SEMANTIC + W_TFIDF + W_FUZZY, 1), "weights must sum to 1"

fuzzy = process.cdist(
    feature_names,
    feature_names,
    scorer=fuzz.token_sort_ratio,
    processor=utils.default_process,
) / 100

#rescaling fuzzy the same way as semantic
fuzzy_baseline = np.median(fuzzy[np.triu_indices(len(feature_names), k=1)])
fuzzy = np.clip((fuzzy - fuzzy_baseline) / (1 - fuzzy_baseline), 0, 1)

similarity = (
    W_SEMANTIC * semantic
    + W_TFIDF *  tfidf_similarity
    + W_FUZZY * fuzzy
)
# A missing description gives no IDF evidence, but every item has zero distance to itself.
np.fill_diagonal(similarity, 1.0)
print(f"Blended similarity matrix: {similarity.shape}")

Blended similarity matrix: (445, 445)


## 3.5 Penalty for conflicting concepts

Embeddings see "Deep winter protection" and "Windproof protection" as similar, but they describe different properties (insulation vs. blocking wind). Pairs where one feature name is only about winter/cold and the other only about wind lose 0.30 similarity (72 pairs). Feature names that mention both are not penalised.


> **TODO:** this rule was carried over from an earlier notebook so we need to check it. We need to compare clustering results with and without the penalty once the evaluation exists

In [6]:
CONCEPTS = {
    "winter": r"\b(winter|cold|freez\w*|frost\w*)\b",
    "windproof": r"\b(wind\w*)\b",
}
CONFLICTS = [("winter", "windproof")]

concepts_by_feature_name = [
    {
        name
        for name, pattern in CONCEPTS.items()
        if re.search(pattern, feature_name, re.IGNORECASE)
    }
    for feature_name in feature_names
]

penalty_matrix = np.zeros_like(similarity)
for first_concept, second_concept in CONFLICTS:
    first_only = np.array(
        [first_concept in concepts and second_concept not in concepts for concepts in concepts_by_feature_name]
    )
    second_only = np.array(
        [second_concept in concepts and first_concept not in concepts for concepts in concepts_by_feature_name]
    )
    penalty_matrix[np.ix_(first_only, second_only)] = PENALTY
    penalty_matrix[np.ix_(second_only, first_only)] = PENALTY

similarity = np.clip(similarity - penalty_matrix, 0, 1)
print(f"Conflicting feature-name pairs penalized: {int(np.count_nonzero(penalty_matrix) // 2)}")

Conflicting attribute pairs penalized: 72


In [7]:

# Save the shared similarity matrix for all clustering notebooks
output_path = Path("data/similarity.npz")
output_path.parent.mkdir(exist_ok=True)

np.savez_compressed(
    output_path,
    similarity=similarity,                 # 445 x 445, values in [0, 1]
    titles=np.array(feature_names),           # row/column order of the matrix
)
print(f"Saved {similarity.shape} similarity matrix to {output_path}")

Saved (445, 445) similarity matrix to data/similarity.npz


# 4. General clustering

All clustering methods use the same distance matrix `D = 1 - S`. The scoring helpers are defined here because the parameter searches in 4.2 and 4.3 already use them; they are explained in 6.1.

In [1]:
import time
from collections import Counter
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from sklearn.cluster import AgglomerativeClustering, DBSCAN, HDBSCAN, KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

with np.load(Path("data/similarity.npz")) as saved:
    S = saved["similarity"].astype(float)
    feature_names = saved["titles"].tolist()

n = len(feature_names)
assert S.shape == (n, n) and np.allclose(S, S.T)
D = np.clip(1 - S, 0, 1)
np.fill_diagonal(D, 0)
G = S.copy()
np.fill_diagonal(G, 0)


def score_labels(labels, distances=D):
    """Scores all feature names; DBSCAN/HDBSCAN noise (-1) becomes one singleton cluster per feature name."""
    labels = np.asarray(labels).copy()
    noise = labels == -1
    n_noise = int(noise.sum())
    full_labels = labels.copy()
    next_label = int(labels[~noise].max() + 1) if np.any(~noise) else 0
    full_labels[noise] = np.arange(next_label, next_label + n_noise)
    silhouette = float("nan")
    if 2 <= len(np.unique(full_labels)) < len(labels):
        silhouette = float(silhouette_score(distances, full_labels, metric="precomputed"))
    return {"labels": labels, "full_labels": full_labels, "clusters": len(set(labels[~noise].tolist())),
            "noise": n_noise, "coverage": float((~noise).mean()), "silhouette_all": silhouette}


def mark_small_clusters_as_noise(labels, minimum_size=2):
    labels = np.asarray(labels).copy()
    for label in np.unique(labels):
        if label != -1 and np.count_nonzero(labels == label) < minimum_size:
            labels[labels == label] = -1
    return labels


def fair_rank(row):
    # selection rule for the density grids: highest silhouette, then highest coverage, then fewer clusters
    return (-row["metrics"]["silhouette_all"], -row["metrics"]["coverage"], row["metrics"]["clusters"])


def medoid_db_like(labels, distances=D):
    """Davies-Bouldin-style score for a precomputed distance matrix, using cluster medoids (lower is better)."""
    cluster_ids = np.unique(labels)
    if len(cluster_ids) < 2:
        return float("nan")
    medoids, scatters = [], []
    for cluster_id in cluster_ids:
        members = np.flatnonzero(labels == cluster_id)
        medoid = members[np.argmin(distances[np.ix_(members, members)].sum(axis=1))]
        medoids.append(medoid)
        scatters.append(float(distances[members, medoid].mean()))
    separation = distances[np.ix_(medoids, medoids)].copy()
    np.fill_diagonal(separation, np.inf)
    scatters = np.asarray(scatters)
    ratios = (scatters[:, None] + scatters[None, :]) / np.maximum(separation, 1e-12)
    return float(ratios.max(axis=1).mean())


def modularity(labels, graph=G):
    """Weighted modularity on the full similarity graph (higher is better)."""
    degree = graph.sum(axis=1)
    two_m = degree.sum()
    same = labels[:, None] == labels[None, :]
    return float(((graph - np.outer(degree, degree) / two_m) * same).sum() / two_m)


clusterings = {}
print(f"Loaded {n} feature names; distance matrix {D.shape}")

Loaded 445 feature names; distance matrix (445, 445)


## 4.1 Agglomerative clustering

In [2]:
AGGLO_THRESHOLD = 0.70  # fixed average-linkage distance threshold, as in 4_agglomerative.ipynb

start = time.perf_counter()
agglo_labels = AgglomerativeClustering(
    n_clusters=None, metric="precomputed", linkage="average", distance_threshold=AGGLO_THRESHOLD
).fit_predict(D)
clusterings["Agglomerative"] = {"metrics": score_labels(agglo_labels), "seconds": time.perf_counter() - start,
                                "params": f"average linkage, threshold={AGGLO_THRESHOLD}"}
print(f"Agglomerative: {clusterings['Agglomerative']['metrics']['clusters']} clusters")

Agglomerative: 138 clusters


## 4.2 DBSCAN

In [3]:
# Grid: eps 0.20-0.68 by 0.02, nine min_samples values; clusters smaller than 2 are relabelled as noise
DBSCAN_EPS = np.round(np.arange(0.20, 0.70, 0.02), 2).tolist()
DBSCAN_MIN_SAMPLES = [2, 3, 4, 5, 6, 8, 10, 12, 14]
dbscan_rows = []
for eps in DBSCAN_EPS:
    for min_samples in DBSCAN_MIN_SAMPLES:
        labels = DBSCAN(eps=eps, min_samples=min_samples, metric="precomputed").fit_predict(D)
        metrics = score_labels(mark_small_clusters_as_noise(labels))
        if metrics["clusters"] >= 2 and np.isfinite(metrics["silhouette_all"]):
            dbscan_rows.append({"eps": eps, "min_samples": min_samples, "metrics": metrics})
dbscan_best = sorted(dbscan_rows, key=fair_rank)[0]

# time one fit with the selected parameters
start = time.perf_counter()
mark_small_clusters_as_noise(DBSCAN(eps=dbscan_best["eps"], min_samples=dbscan_best["min_samples"], metric="precomputed").fit_predict(D))
clusterings["DBSCAN"] = {"metrics": dbscan_best["metrics"], "seconds": time.perf_counter() - start,
                         "params": f"eps={dbscan_best['eps']:.2f}, min_samples={dbscan_best['min_samples']}"}
print(f"DBSCAN grid: {len(DBSCAN_EPS) * len(DBSCAN_MIN_SAMPLES)} combinations, {len(dbscan_rows)} valid; "
      f"selected {clusterings['DBSCAN']['params']}")

dbscan_grid = np.full((len(DBSCAN_MIN_SAMPLES), len(DBSCAN_EPS)), np.nan)
for row in dbscan_rows:
    dbscan_grid[DBSCAN_MIN_SAMPLES.index(row["min_samples"]), DBSCAN_EPS.index(row["eps"])] = row["metrics"]["silhouette_all"]
figure = go.Figure(go.Heatmap(z=dbscan_grid, x=DBSCAN_EPS, y=DBSCAN_MIN_SAMPLES, colorscale="Viridis",
                              colorbar={"title": "Silhouette"},
                              hovertemplate="eps=%{x}, min_samples=%{y}<br>silhouette=%{z:.3f}<extra></extra>"))
figure.update_layout(template="simple_white", width=700, height=420, title="DBSCAN: eps × min_samples (all-feature-name silhouette)",
                     xaxis_title="eps", yaxis_title="min_samples")
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

DBSCAN grid: 225 combinations, 129 valid; selected eps=0.40, min_samples=2


## 4.3 HDBSCAN

Note: the selected values lie at the low edge of the tested grid.

In [4]:
# Grid: each pair is tested only when min_samples <= min_cluster_size
HDBSCAN_MIN_CLUSTER_SIZE = [2, 3, 4, 5, 8, 12, 20]
HDBSCAN_MIN_SAMPLES = [2, 3, 4, 5, 6, 8]
hdbscan_rows = []
for min_cluster_size in HDBSCAN_MIN_CLUSTER_SIZE:
    for min_samples in HDBSCAN_MIN_SAMPLES:
        if min_samples > min_cluster_size:
            continue
        labels = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, metric="precomputed", copy=True).fit_predict(D)
        metrics = score_labels(labels)
        if metrics["clusters"] >= 2 and np.isfinite(metrics["silhouette_all"]):
            hdbscan_rows.append({"min_cluster_size": min_cluster_size, "min_samples": min_samples, "metrics": metrics})
hdbscan_best = sorted(hdbscan_rows, key=fair_rank)[0]

start = time.perf_counter()
HDBSCAN(min_cluster_size=hdbscan_best["min_cluster_size"], min_samples=hdbscan_best["min_samples"],
        metric="precomputed", copy=True).fit_predict(D)
clusterings["HDBSCAN"] = {"metrics": hdbscan_best["metrics"], "seconds": time.perf_counter() - start,
                          "params": f"min_cluster_size={hdbscan_best['min_cluster_size']}, min_samples={hdbscan_best['min_samples']}"}
print(f"HDBSCAN grid: {len(hdbscan_rows)} valid pairs; selected {clusterings['HDBSCAN']['params']}")

hdbscan_grid = np.full((len(HDBSCAN_MIN_SAMPLES), len(HDBSCAN_MIN_CLUSTER_SIZE)), np.nan)
for row in hdbscan_rows:
    hdbscan_grid[HDBSCAN_MIN_SAMPLES.index(row["min_samples"]), HDBSCAN_MIN_CLUSTER_SIZE.index(row["min_cluster_size"])] = row["metrics"]["silhouette_all"]
figure = go.Figure(go.Heatmap(z=hdbscan_grid, x=HDBSCAN_MIN_CLUSTER_SIZE, y=HDBSCAN_MIN_SAMPLES, colorscale="Viridis",
                              colorbar={"title": "Silhouette"},
                              hovertemplate="min_cluster_size=%{x}, min_samples=%{y}<br>silhouette=%{z:.3f}<extra></extra>"))
figure.update_layout(template="simple_white", width=700, height=420, title="HDBSCAN: min_cluster_size × min_samples (all-feature-name silhouette)",
                     xaxis_title="min_cluster_size", yaxis_title="min_samples")
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

HDBSCAN grid: 26 valid pairs; selected min_cluster_size=2, min_samples=2


# 5. Graph clustering

## 5.1 General spectral clustering

In [5]:
# Same pipeline as 5_spectral.ipynb: 15-nearest-neighbour graph, normalized Laplacian, k = 35 (chosen there in section 5.2)
K_NEIGHBORS = 15
K_CLUSTERS = 35

start = time.perf_counter()
W = G.copy()
thresholds = np.sort(W, axis=1)[:, -K_NEIGHBORS][:, None]
W = np.where((W >= thresholds) | (W >= thresholds.T), W, 0)
inverse_sqrt_degree = 1 / np.sqrt(np.maximum(W.sum(axis=1), 1e-12))
laplacian = np.eye(n) - W * np.outer(inverse_sqrt_degree, inverse_sqrt_degree)
_, eigenvectors = np.linalg.eigh(laplacian)
U = eigenvectors[:, :K_CLUSTERS]
U = U / np.maximum(np.linalg.norm(U, axis=1, keepdims=True), 1e-12)
spectral_labels = KMeans(n_clusters=K_CLUSTERS, n_init=10, random_state=0).fit_predict(U)
clusterings["Spectral"] = {"metrics": score_labels(spectral_labels), "seconds": time.perf_counter() - start,
                           "params": f"k={K_CLUSTERS}, k_neighbors={K_NEIGHBORS}, seed=0"}
print(f"Spectral: {K_CLUSTERS} clusters on a graph with {int(np.count_nonzero(np.triu(W)))} edges")

Spectral: 35 clusters on a graph with 4618 edges


## 5.2 Filtered spectral clustering by product type

# 6. Evaluation

## 6.1 Setup

All methods are scored on the same distance matrix `D = 1 - S` with the helpers from section 4. The measures are **internal**: they describe the shape of the clusters, not whether two feature names really mean the same.

- **Silhouette** (higher is better, −1 to 1): whether a feature name is closer to its own cluster than to the nearest other cluster, averaged over all feature names.
- **Medoid Davies–Bouldin-like score** (lower is better): for each cluster, the worst ratio of (scatter of two clusters) to (distance between their medoids), averaged. It is an adaptation for a precomputed distance matrix, not the standard index.
- **Modularity** (higher is better): within-cluster similarity compared with what random links would give, on the full similarity graph.
- **Noise and coverage:** to score all 445 feature names, each DBSCAN/HDBSCAN noise feature name counts as its own singleton cluster. A singleton has zero silhouette and zero scatter, which can make the Davies–Bouldin-like score look better. **Coverage** is the share of feature names in a non-noise cluster and must be read together with the scores.
- **Runtime:** one fit with the selected parameters.

| Method | Parameters tested | Selection |
|---|---|---|
| Agglomerative | average linkage, fixed `distance_threshold = 0.70` | fixed |
| DBSCAN | `eps` 0.20–0.68 by 0.02 × `min_samples` ∈ {2,3,4,5,6,8,10,12,14} (225 runs); clusters < 2 marked as noise | highest silhouette, then coverage, then fewer clusters |
| HDBSCAN | `min_cluster_size` ∈ {2,3,4,5,8,12,20} × `min_samples` ∈ {2,3,4,5,6,8}, skipping `min_samples > min_cluster_size` | same rule as DBSCAN |
| Spectral | 15 nearest neighbours; k swept 10–100 by 5 in `5_spectral.ipynb` | k = 35 (sweep and inspection) |

## 6.2 Comparison of all methods

In [6]:
rows = []
for name, result in clusterings.items():
    metrics = result["metrics"]
    rows.append({
        "method": name,
        "parameters": result["params"],
        "clusters": metrics["clusters"],
        "coverage": f"{metrics['coverage']:.1%}",
        "silhouette": round(metrics["silhouette_all"], 3),
        "medoid_db_like": round(medoid_db_like(metrics["full_labels"]), 3),
        "modularity": round(modularity(metrics["full_labels"]), 3),
        "runtime_s": round(result["seconds"], 3),
    })

import pandas as pd
comparison = pd.DataFrame(rows).set_index("method")
comparison

,parameters,clusters,coverage,silhouette,medoid_db_like,modularity,runtime_s
method,,,,,,,
Agglomerative,"average linkage, threshold=0.7",138,100.0%,0.184,0.911,0.070,0.040
DBSCAN,"eps=0.40, min_samples=2",59,37.1%,0.138,0.492,0.016,0.005
HDBSCAN,"min_cluster_size=2, min_samples=2",106,73.3%,0.178,0.725,0.030,0.018
Spectral,"k=35, k_neighbors=15, seed=0",35,100.0%,0.137,1.585,0.104,0.145


## 6.3 Qualitative examples

Each panel shows the same t-SNE projection, coloured by one method's clusters; noise feature names are grey crosses. t-SNE is for display only and can distort distances.

In [7]:
embedding = TSNE(n_components=2, metric="precomputed", init="random", learning_rate="auto",
                 perplexity=30, random_state=0).fit_transform(D)

figure = make_subplots(rows=2, cols=2, subplot_titles=[
    f"{name}: {result['metrics']['clusters']} clusters, silhouette {result['metrics']['silhouette_all']:.3f}"
    for name, result in clusterings.items()])
for index, (name, result) in enumerate(clusterings.items()):
    row, col = index // 2 + 1, index % 2 + 1
    labels = result["metrics"]["labels"]
    noise = labels == -1
    figure.add_trace(go.Scattergl(
        x=embedding[~noise, 0], y=embedding[~noise, 1], mode="markers",
        marker={"color": labels[~noise], "colorscale": "Turbo", "size": 7, "opacity": 0.8, "showscale": False},
        text=np.asarray(feature_names)[~noise], hovertemplate="%{text}<extra>cluster</extra>", showlegend=False),
        row=row, col=col)
    if noise.any():
        figure.add_trace(go.Scattergl(
            x=embedding[noise, 0], y=embedding[noise, 1], mode="markers",
            marker={"color": "#999999", "size": 6, "symbol": "x", "opacity": 0.65},
            text=np.asarray(feature_names)[noise], hovertemplate="%{text}<extra>noise</extra>", showlegend=False),
            row=row, col=col)
figure.update_xaxes(showticklabels=False)
figure.update_yaxes(showticklabels=False)
figure.update_layout(template="simple_white", height=850, width=1100, title="Feature-name clusters per method (shared t-SNE projection)")
display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

## 6.4 General vs filtered spectral clustering

## 6.5 Summary

# 7. Limitations and future work